<a href="https://colab.research.google.com/github/Angela-Benedict/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [79]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [80]:
df['revenue'] = df['qty'] * df['price']
print('Rows:', len(df))
print(
    'total revenue: ', sum(df['revenue']),
    'total units: ', sum(df['qty'])
)

Rows: 400
total revenue:  8520.0 total units:  783


I added a column called 'revenue' and checked that the dataframe now has 400 rows. 8520.0 is the total revenue and the sum of all the revenue numbers in the revenue column.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [81]:
total_revenue = sum(df['revenue'])
by_category = df.groupby('category')['revenue'].sum().reset_index()
by_category.sort_values(by='revenue', ascending= False)
by_category['share_of_total (%)'] = ((by_category['revenue']/total_revenue)*100).round(2)
by_category.head()

,category,revenue,share_of_total (%)
0,Drink,1554.0,18.24
1,Food,4293.0,50.39
2,Merch,1771.5,20.79
3,RainGear,901.5,10.58


This table contains revenue for each category item and the share as a percentage that each item is responsible for of the total revenue.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [82]:
vendor_summary = (
    df.groupby('vendor_id')
      .agg(
          avg_order_revenue = ('revenue', 'mean'),
          order_count = ('revenue', 'count')
      )
      .sort_values('avg_order_revenue', ascending = False)
)
print(vendor_summary)

           avg_order_revenue  order_count
vendor_id                                
V-01               22.595745           94
V-18               21.750000          108
V-05               20.580645           93
V-10               20.314286          105


Based on this data, vendor V-01 has the highest average order revenue of about 22.596 from 94 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [83]:
df[df['category']=='Merch']['revenue']
Merch_revenue_total = sum(df[df['category']=='Merch']['revenue'])
Merch_revenue_percent = Merch_revenue_total / (sum(df['revenue']))
print(f"{Merch_revenue_percent:.1%}")

20.8%


This means that 20.8% of revenue gained across all vendors is attributed to the selling of merch products.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [84]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined_df = df.merge(vendor_names, how = 'left', validate = 'many_to_one')
joined_df['vendor_name'] = joined_df['vendor_name'].fillna('Unknown vendor')
print('Before:', len(df), 'After:', len(joined_df))
print('Before:', sum(df['revenue']), 'After:', sum(joined_df['revenue']))
joined_df.head()

# TODO: merge, validate, and report the unmatched vendor

Before: 400 After: 400
Before: 8520.0 After: 8520.0


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown vendor
2,V-18,Drink,3,4.5,13.5,Unknown vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown vendor


**The unmatched vendor, and what I did about it:** I called the unmatched vendor "unknown vendor" used this to fill the NaN's because I wanted to keep the rows and the total_revenue the same.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [85]:
pivot_df_table = joined_df.pivot_table(index = 'vendor_name',
                                 columns = 'category',
                                 values = 'revenue')
pivot_df_table

category,Drink,Food,Merch,RainGear
vendor_name,,,,
Cav Merch North,17.327586,20.676471,25.031250,19.50
Hoos Burgers,14.250000,24.327273,21.970588,24.15
Rotunda Tacos,17.558824,23.837838,20.375000,16.30
Unknown vendor,18.774194,23.686047,23.113636,20.00


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [86]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined_df) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell these vendors to sell more food instead of raingear. Based on share of total revenue across all vendors, food made up 50.39% of total revenue while raingear only made up a smaller 10.58%. This makes sense, as food is a necessity for customers at a game and are therefor emore likely to be sold. I would also keep merch products interesting and available for purchase because vendors made more profit off merch than drinks, with merch making up 20.79% of total revenue and drinks making up 18.24%.

b) My pivot table from Q6 is least trustworthy because it includes the unmatched vendor which may change how the data is viewed/interpreted and the revenue numbers don't seem to add up to the total revenue.